# 4) Congestion and nodal prices — Exercises

**Goals:** reproduce DC flows from the PTDFs, decompose nodal prices line by line, follow cost, rent and prices as lines tighten, value a line reinforcement with its shadow price, and explain nodal prices outside the range of every generator's cost.

The setup cell holds the lecture's network, plants and `PTDF`, the generator-to-bus map `G`, the lecture's `dispatch(limits)` — now accepting one limit for all lines or a dict `{line: MW}` (other lines unlimited) — and `pypsa_network(limits)`. Power in MW, prices in €/MWh, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

warnings.filterwarnings("ignore")
for name in ("pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

# ---- the lecture's network, plants and PTDFs ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM = 0.32
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")


def ptdf(reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(LINES), len(BUSES)))
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))                            # generator-to-bus map
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1


def as_limits(limits):
    """A number (every line) or a dict {line: MW} (others unlimited) -> Series of limits."""
    if isinstance(limits, (int, float)):
        return pd.Series(float(limits), index=LINES.index)
    return pd.Series({line: float(limits.get(line, 1e5)) for line in LINES.index})


def dispatch(limits):
    """The lecture's DC-constrained dispatch, with one limit for all lines or a limit per line."""
    limits = as_limits(limits)
    gi = pd.Index(GENS.index, name="gen")
    flow_per_gen, flow_of_load = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[gi]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray(flow_per_gen[l], coords=[gi])).sum()
        m.add_constraints(flow <= limits[line] + flow_of_load[l], name=f"upper {line}")
        m.add_constraints(flow >= -limits[line] + flow_of_load[l], name=f"lower {line}")
    m.add_objective((p * xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[gi])).sum())
    m.solve(solver_name="highs", output_flag=False)
    lam = float(m.constraints["balance"].dual)
    mu = pd.Series({line: float(m.constraints[f"upper {line}"].dual) + float(m.constraints[f"lower {line}"].dual)
                    for line in LINES.index})
    p_sol = p.solution.to_pandas()
    flows = pd.Series(PTDF.to_numpy() @ (G @ p_sol.to_numpy() - LOAD.to_numpy()), index=LINES.index)
    return {"cost": float(m.objective.value), "dispatch": p_sol, "flows": flows, "lmp": lam + PTDF.T @ mu,
            "lambda": lam, "mu": mu, "limits": limits}


def pypsa_network(limits):
    """The same network in PyPSA (call .optimize() or .lpf() on it)."""
    limits = as_limits(limits)
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    for bus, load in LOAD.items():
        if load:
            n.add("Load", f"{bus} load", bus=bus, p_set=load)
    for name, g in GENS.iterrows():
        n.add("Generator", name, bus=g.bus, p_nom=g.p_max_MW, marginal_cost=g.cost_EUR_per_MWh)
    for name, line in LINES.iterrows():
        n.add("Line", name, bus0=line.bus0, bus1=line.bus1, x=X_KM * line.km, r=0.0, s_nom=limits[name])
    return n

### Warm-ups

**1. Flows from the PTDFs**

Write `ptdf_flows(dispatch_mw)` returning the line flows (MW) caused by a dispatch (a Series over `GENS.index`) and the loads. Check it on the lecture's uncongested dispatch against PyPSA's DC power flow (`lpf()`) with the generators fixed at that dispatch.

In [2]:
def ptdf_flows(dispatch_mw: pd.Series) -> pd.Series:
    """Line flows (MW) = PTDF x (generation - load) at each bus."""
    ### BEGIN SOLUTION
    injection = G @ dispatch_mw[GENS.index].to_numpy() - LOAD.to_numpy()
    return pd.Series(PTDF.to_numpy() @ injection, index=LINES.index)
    ### END SOLUTION

In [3]:
free = dispatch(2000)
n = pypsa_network(2000)
n.generators.p_set = free["dispatch"][n.generators.index]
n.lpf()
assert np.abs(ptdf_flows(free["dispatch"]).to_numpy() - n.lines_t.p0.iloc[0].to_numpy()).max() < 1e-6
ptdf_flows(free["dispatch"]).round(1)

Zagreb-Rijeka    -390.9
Rijeka-Velebit   -490.9
Velebit-Split     109.1
Zagreb-Split     -273.1
Zagreb-Osijek    -236.0
Osijek-Split      -86.0
dtype: float64

**2. Where a nodal price comes from**

Write `decompose(res)` returning a table with one row per bus: λ, the contribution PTDF_ℓn · μ_ℓ of each **congested** line, and their sum. Check the sum against `res["lmp"]` and against PyPSA's nodal prices for the lecture's 400 MW case.

In [4]:
def decompose(res: dict) -> pd.DataFrame:
    """lambda + per-line contributions PTDF[l, n] * mu[l] for the congested lines, and the total."""
    ### BEGIN SOLUTION
    congested = res["mu"].index[res["mu"].abs() > 1e-6]
    table = pd.DataFrame({"lambda": res["lambda"]}, index=BUSES)
    for line in congested:
        table[line] = PTDF.loc[line] * res["mu"][line]
    table["LMP"] = table.sum(axis=1)
    return table
    ### END SOLUTION

In [5]:
res400 = dispatch(400)
parts = decompose(res400)
n = pypsa_network(400)
n.optimize(solver_name="highs", log_to_console=False, progress=False)
assert np.abs(parts["LMP"].to_numpy() - res400["lmp"].to_numpy()).max() < 1e-9
assert np.abs(parts["LMP"].to_numpy() - n.buses_t.marginal_price.iloc[0][BUSES].to_numpy()).max() < 1e-6
parts.round(2)

         lambda  Zagreb-Rijeka  Rijeka-Velebit     LMP
Zagreb    130.0           0.00            0.00  130.00
Rijeka    130.0         -67.67           27.67   90.00
Split     130.0         -29.31          -43.90   56.80
Osijek    130.0         -12.63          -18.92   98.45
Velebit   130.0         -52.04          -77.96    0.00

### Core

**3. Tightening the lines**

For a common limit of [1000, 600, 500, 450, 400, 300] MW on every line, report: system cost, congestion rent, curtailed wind, number of congested lines, and the spread between the highest and lowest nodal price. Write `limit_sweep(limits)`.

In [6]:
def limit_sweep(limits=(1000, 600, 500, 450, 400, 300)) -> pd.DataFrame:
    """Cost, rent, curtailment, congested lines and price spread for each common line limit."""
    ### BEGIN SOLUTION
    rows = {}
    for limit in limits:
        r = dispatch(limit)
        rent = float((r["lmp"] * LOAD).sum() - sum(r["lmp"][GENS.bus[g]] * r["dispatch"][g] for g in GENS.index))
        rows[limit] = {"cost_EUR_per_h": r["cost"], "rent_EUR_per_h": rent,
                       "wind_curtailed_MW": GENS.p_max_MW["wind Velebit"] - r["dispatch"]["wind Velebit"],
                       "congested_lines": int((r["mu"].abs() > 1e-6).sum()),
                       "price_spread_EUR_per_MWh": r["lmp"].max() - r["lmp"].min()}
    return pd.DataFrame(rows).T.rename_axis("line limit (MW)")
    ### END SOLUTION

In [7]:
sweep = limit_sweep()
assert sweep.cost_EUR_per_h.is_monotonic_increasing
assert sweep.loc[1000, "congested_lines"] == 0 and np.isclose(sweep.loc[1000, "rent_EUR_per_h"], 0, atol=1e-6)
sweep.round(1)

                 cost_EUR_per_h  rent_EUR_per_h  wind_curtailed_MW  \
line limit (MW)                                                      
1000                    71000.0             0.0                0.0   
600                     71000.0             0.0                0.0   
500                     71000.0             0.0                0.0   
450                     72846.2         78692.3               10.6   
400                     81956.3         86077.7              115.0   
300                    103475.7         64558.3              280.6   

                 congested_lines  price_spread_EUR_per_MWh  
line limit (MW)                                             
1000                         0.0                       0.0  
600                          0.0                       0.0  
500                          0.0                       0.0  
450                          2.0                     105.6  
400                          2.0                     130.0  
300         

**4. What is a reinforcement worth?**

At the 400 MW limit, raise the limit of **one** congested line at a time — by 1 MW and by 100 MW, all other lines at 400 MW — and compare the cost saving with the line's shadow price: |μ| × 1 MW and |μ| × 100 MW. Write `reinforcement_value()`.

In [8]:
def reinforcement_value() -> pd.DataFrame:
    """Cost saving of +1 MW and +100 MW on each congested line, against |mu| times the increase."""
    ### BEGIN SOLUTION
    base = dispatch(400)
    rows = {}
    for line in base["mu"].index[base["mu"].abs() > 1e-6]:
        row = {"abs_mu_EUR_per_MWh": abs(base["mu"][line])}
        for extra in (1, 100):
            limits = {l: 400.0 for l in LINES.index}
            limits[line] = 400.0 + extra
            row[f"saving_+{extra}MW"] = base["cost"] - dispatch(limits)["cost"]
            row[f"mu_x_{extra}MW"] = abs(base["mu"][line]) * extra
        rows[line] = row
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [9]:
value = reinforcement_value()
assert np.allclose(value["saving_+1MW"], value["mu_x_1MW"], rtol=1e-6), "the shadow price is the value of the first MW"
value.round(1)

                abs_mu_EUR_per_MWh  saving_+1MW  mu_x_1MW  saving_+100MW  \
Zagreb-Rijeka                 86.1         86.1      86.1         5680.7   
Rijeka-Velebit               129.0        129.0     129.0        10956.3   

                mu_x_100MW  
Zagreb-Rijeka       8614.7  
Rijeka-Velebit     12904.7  

**5. Interpretation**

Why are the nodal prices identical at 400 and 300 MW although cost and dispatch change? What does the sweep show about congestion rent as lines tighten? How well does the shadow price predict the value of +1 MW and of +100 MW, and why does it fail for the larger step?

<!-- BEGIN SOLUTION -->
Nodal prices are set by the **marginal units and the congested lines**, not by the size of the limits. At 400 and 300 MW the same two lines are congested and the same three units are marginal — wind at Velebit, gas at Rijeka, imports at Zagreb — so every price is the same combination of the same costs, through the same PTDFs. Tightening the limit changes how much each unit produces (more gas and imports, 281 MW of wind curtailed instead of 115) and so the cost, but not the prices. Prices only jump when the set of binding lines or marginal units changes, as between 450 and 400 MW, where imports become marginal at Zagreb and its price rises from 105.64 to 130 €/MWh.

The congestion rent does **not** grow steadily as lines tighten. It rises from zero to 78,692 €/h at 450 MW and 86,078 €/h at 400 MW, then falls to 64,558 €/h at 300 MW. The rent is the sum of |μ| × limit over the congested lines; between 400 and 300 MW the shadow prices are unchanged, so the rent shrinks in proportion to the limit (86,078 × 300/400 = 64,558). Less power crosses the congested lines, so less is collected on them, even though the system cost keeps rising — one more reason not to read the rent as a measure of the damage congestion does.

The shadow price is **exact for the first MW**: +1 MW on Zagreb–Rijeka saves 86.1 €/h and on Rijeka–Velebit 129.0 €/h, their |μ| to the cent. For +100 MW it overstates the saving — 5,681 instead of 8,615 €/h on Zagreb–Rijeka, 10,956 instead of 12,905 €/h on Rijeka–Velebit. A shadow price is a marginal value: it holds only as long as the same constraints bind. Raising Rijeka–Velebit to 500 MW removes the congestion altogether (Zagreb–Rijeka then carries 391 MW, below its limit), so the saving stops at the full cost of congestion, 10,956 €/h. To value a real reinforcement, re-solve the model with it; the shadow price tells you where to look first.
<!-- END SOLUTION -->

### Challenge

**6. Prices no generator would offer**

Every generator here costs between 0 and 130 €/MWh. Run two cases with **only one** line limited (all others unlimited): (a) Zagreb–Rijeka at 200 MW, (b) Rijeka–Velebit at 250 MW. Report all nodal prices and decompose Rijeka's price; confirm both cases in PyPSA. Write `outside_the_range()`.

In [10]:
def outside_the_range() -> dict:
    """Nodal prices and Rijeka's decomposition for the two single-line cases."""
    ### BEGIN SOLUTION
    out = {}
    for label, limits in {"(a) Zagreb-Rijeka at 200 MW": {"Zagreb-Rijeka": 200},
                          "(b) Rijeka-Velebit at 250 MW": {"Rijeka-Velebit": 250}}.items():
        r = dispatch(limits)
        n = pypsa_network(limits)
        n.optimize(solver_name="highs", log_to_console=False, progress=False)
        assert np.abs(n.buses_t.marginal_price.iloc[0][BUSES].to_numpy() - r["lmp"].to_numpy()).max() < 1e-6
        out[label] = {"lmp": r["lmp"], "rijeka": decompose(r).loc["Rijeka"], "dispatch": r["dispatch"]}
    return out
    ### END SOLUTION

In [11]:
cases = outside_the_range()
prices = pd.DataFrame({k: v["lmp"] for k, v in cases.items()})
assert prices.loc["Rijeka"].min() < 0 and prices.loc["Rijeka"].max() > 130
print("Rijeka's price, decomposed:")
print(pd.DataFrame({k: v["rijeka"] for k, v in cases.items()}).round(2).fillna(""))
prices.round(2)

Rijeka's price, decomposed:
               (a) Zagreb-Rijeka at 200 MW (b) Rijeka-Velebit at 250 MW
LMP                                 -39.05                       143.14
Rijeka-Velebit                                                     37.5
Zagreb-Rijeka                      -169.05                             
lambda                               130.0                       105.64


         (a) Zagreb-Rijeka at 200 MW  (b) Rijeka-Velebit at 250 MW
Zagreb                        130.00                        105.64
Rijeka                        -39.05                        143.14
Split                          56.80                         46.15
Osijek                         98.45                         80.00
Velebit                         0.00                          0.00

**7. Interpretation**

Explain in words why one more MW of load at Rijeka is worth −39 €/MWh in case (a) and costs 143 €/MWh in case (b). What would a generator at Rijeka be paid in case (a), and what does that say about where new generation should — or should not — be built?

<!-- BEGIN SOLUTION -->
In case (a) only Zagreb–Rijeka is limited, and it carries its 200 MW from Rijeka towards Zagreb. Rijeka's gas plant is off; the marginal units are wind at Velebit (544 of 600 MW) and imports at Zagreb (256 MW). One more MW of load at Rijeka is consumed **behind** the congested line: it absorbs power that would otherwise push towards Zagreb, and so frees room on the line. The optimum uses that room for more cheap wind — re-solving with one more MW at Rijeka gives 1.30 MW more wind at Velebit and 0.30 MW less import at Zagreb. Wind covers the extra MW and replaces some of the 130 €/MWh imports as well, so the system as a whole saves money when Rijeka consumes more: its price is −39.05 €/MWh, which the formula writes as λ = 130 minus 169.05 from the congested line. A generator at Rijeka would have to **pay** 39 €/MWh to produce, because every MW it adds makes the congestion worse.

In case (b) only Rijeka–Velebit is limited, at 250 MW towards Rijeka. Gas at Rijeka already runs at its full 600 MW; the marginal units are wind at Velebit (389 MW) and coal at Osijek (211 MW). One more MW at Rijeka cannot come from Velebit — that line is full — and cannot come from Rijeka's own plant. It has to arrive over Zagreb–Rijeka, and the extra coal flowing in from Osijek loops partly over Velebit, so wind must be cut back to keep the congested line within its limit. Re-solving with one more MW at Rijeka gives 1.79 MW more coal at Osijek and 0.79 MW less wind: every MW delivered to Rijeka needs 1.79 MW of coal at 80 €/MWh, which is 143.14 €/MWh — more than any plant's offer.

The lesson for investment is direct: nodal prices tell you where generation and consumption are worth most. In case (a) new generation at Rijeka would be harmful, while new consumption there — an electrolyser, storage charging, an industrial load — would be paid to come. In case (b) the opposite holds. A single zonal price hides both signals, which is the debate of lesson 5.
<!-- END SOLUTION -->